In [24]:

# 1. Forcefully wipe out the local unmounted ghost directory
!rm -rf /content/drive

# 2. Re-create a pristine, completely empty mount directory
import os
os.makedirs('/content/drive', exist_ok=True)

# 3. Mount fresh
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [25]:
# Install dependencies if you are running this in a fresh runtime environment
!pip install --quiet pymupdf pillow matplotlib

In [26]:
import os
import fitz  # PyMuPDF
from PIL import Image
import logging
from typing import List, Dict, Any, Tuple

# Google Drive project folder
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/Insurance-Claim-AI-Learning"

# Set up clean logging patterns for production tracking
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("PDF_Ingestion")

class PDFProcessor:
    """
    A production-ready utility class to handle safe document ingestion,
    validation, and high-fidelity image rasterization.
    """
    def __init__(self, output_dir: str = os.path.join(PROJECT_DIR, "storage", "rasterized")):
        self.output_dir = output_dir
        os.makedirs(self.output_dir, exist_ok=True)

    def validate_and_open(self, pdf_path: str) -> Tuple[fitz.Document, Dict[str, Any]]:
        """
        Safely checks file existence, opens the document, and extracts metadata.
        Raises descriptive exceptions to map directly to FastAPI HTTP status codes.
        """
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f"Target document missing: {pdf_path}")

        try:
            doc = fitz.open(pdf_path)
        except Exception as e:
            raise ValueError(f"Corrupted or invalid PDF structure: {str(e)}")

        if doc.is_encrypted:
            raise PermissionError("Process aborted: Target document is password protected.")

        metadata = {
            "page_count": len(doc),
            "is_encrypted": doc.is_encrypted,
            "metadata_raw": doc.metadata
        }

        logger.info(f"Successfully validated document. Pages detected: {metadata['page_count']}")
        return doc, metadata

    def rasterize_page(self, doc: fitz.Document, page_num: int, dpi: int = 200) -> str:
        """
        Converts a specific page index into a high-fidelity image string path.
        Standardizes resolutions using DPI scale matrices.
        """
        if page_num < 0 or page_num >= len(doc):
            raise IndexError(f"Page index {page_num} out of bounds for document.")

        page = doc[page_num]

        # Calculate optimal transformation matrix based on targeted DPI.
        # Default PDF resolution is 72 points per inch. 200 DPI gives excellent balance of OCR clarity and file size.
        zoom = dpi / 72
        matrix = fitz.Matrix(zoom, zoom)

        # Render page to an RGB pixel buffer map
        pix = page.get_pixmap(matrix=matrix, alpha=False)

        # Save output image
        output_filename = f"page_{page_num + 1}_dpi{dpi}.png"
        output_path = os.path.join(self.output_dir, output_filename)

        pix.save(output_path)
        return output_path

In [27]:
import os
import fitz  # PyMuPDF

# Google Drive project folder
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/Insurance-Claim-AI-Learning"

# Create project folder if it doesn't exist
os.makedirs(PROJECT_DIR, exist_ok=True)

# PDF path
PDF_PATH = os.path.join(PROJECT_DIR, "mock_claim.pdf")


def create_mock_insurance_pdf(filename: str = PDF_PATH):
    """
    Generates a dummy multi-page PDF containing sample claim documentation.
    """

    doc = fitz.open()

    # -----------------------------
    # Page 1: Claim Form Cover
    # -----------------------------
    page1 = doc.new_page()

    page1.insert_text(
        (50, 100),
        "AUTO INSURANCE CLAIM REPORT",
        fontsize=24,
        color=(0, 0, 0)
    )

    page1.insert_text(
        (50, 150),
        "Policyholder: Jane Doe",
        fontsize=14
    )

    page1.insert_text(
        (50, 180),
        "Claim ID: #987123-A",
        fontsize=14
    )

    page1.insert_text(
        (50, 210),
        "Date of Accident: 2026-05-14",
        fontsize=14
    )

    # -----------------------------
    # Page 2: Medical Bill
    # -----------------------------
    page2 = doc.new_page()

    page2.insert_text(
        (50, 80),
        "ITEMIZED MEDICAL BILLING",
        fontsize=18,
        color=(0.8, 0, 0)
    )

    page2.insert_text(
        (50, 130),
        "1. Emergency Room Evaluation ....... $1,200.00",
        fontsize=12
    )

    page2.insert_text(
        (50, 160),
        "2. X-Ray (Right Wrist Full View) ..... $450.00",
        fontsize=12
    )

    page2.insert_text(
        (50, 190),
        "3. Wrist Splint & Medication ......... $150.00",
        fontsize=12
    )

    page2.insert_text(
        (50, 230),
        "TOTAL DUE: $1,800.00",
        fontsize=14
    )

    # Save PDF
    doc.save(filename)
    doc.close()

    print(f"✅ PDF saved to:\n{filename}")


# Execute creation
create_mock_insurance_pdf()

✅ PDF saved to:
/content/drive/MyDrive/Colab Notebooks/Insurance-Claim-AI-Learning/mock_claim.pdf


In [28]:
# Instantiate our system processor component
processor = PDFProcessor()

# Step 1: Open and audit file
doc, info = processor.validate_and_open(PDF_PATH)
print(f"\n--- Extraction Metadata ---\n{info}\n---------------------------")

# Step 2: Loop and extract images dynamically
generated_images = []
for i in range(info["page_count"]):
    img_path = processor.rasterize_page(doc, page_num=i, dpi=200)
    generated_images.append(img_path)
    print(f"Generated Render Output -> {img_path}")

# Always close document contexts when processing completes to avoid system locks
doc.close()


--- Extraction Metadata ---
{'page_count': 2, 'is_encrypted': False, 'metadata_raw': {'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': '', 'producer': '', 'creationDate': '', 'modDate': '', 'trapped': '', 'encryption': None}}
---------------------------
Generated Render Output -> /content/drive/MyDrive/Colab Notebooks/Insurance-Claim-AI-Learning/storage/rasterized/page_1_dpi200.png
Generated Render Output -> /content/drive/MyDrive/Colab Notebooks/Insurance-Claim-AI-Learning/storage/rasterized/page_2_dpi200.png


In [29]:
def production_stream_pipeline(pdf_path: str, target_dpi: int = 150):
    """
    A generator function that streams document pages sequentially.
    Keeps the operational memory footprint completely flat regardless of PDF size.
    """
    processor = PDFProcessor()
    doc, info = processor.validate_and_open(pdf_path)

    for page_idx in range(info["page_count"]):
        logger.info(f"Processing chunk allocation stream for page index {page_idx + 1}")
        img_file = processor.rasterize_page(doc, page_num=page_idx, dpi=target_dpi)

        # Yield runtime control back to calling process loop
        yield page_idx + 1, img_file

        # Optional: You could delete the file here after downstream components use it
        # to save local disk space.

    doc.close()
    logger.info("Streaming memory context freed.")

# Test execution of our stream loop
for page_num, image_file_path in production_stream_pipeline(PDF_PATH):
    print(f"📡 API Layer received stream -> Page {page_num}: Ready for model processing at {image_file_path}")

📡 API Layer received stream -> Page 1: Ready for model processing at /content/drive/MyDrive/Colab Notebooks/Insurance-Claim-AI-Learning/storage/rasterized/page_1_dpi150.png
📡 API Layer received stream -> Page 2: Ready for model processing at /content/drive/MyDrive/Colab Notebooks/Insurance-Claim-AI-Learning/storage/rasterized/page_2_dpi150.png
